# Stacking Ensemble — Apparatus Expert Split

3-stage pipeline for Gym99 action recognition with extreme class imbalance:

| Stage | What happens |
|---|---|
| **1 — Expert Training** | Train 4 ST-GCN models, one per apparatus (VT / FX / BB / UB), using `train_gym99.py --apparatus` flag |
| **2 — Feature Extraction** | Pass all samples through the 4 frozen experts; collect 256-dim vector before final FC; concat → 1024-dim super-vector |
| **3 — Meta-Learner** | Train a lightweight MLP `1024 → 512 → Dropout(0.5) → 256 → 99`; evaluate with Confusion Matrix |

**Apparatus label mapping:**
```
VT  Vault           Clabel  0 –  5  (  6 classes)
FX  Floor Exercise  Clabel  6 – 40  ( 35 classes)
BB  Balance Beam    Clabel 41 – 73  ( 33 classes)
UB  Uneven Bars     Clabel 74 – 98  ( 25 classes)
```

In [ ]:
# ── Cell 1: Environment & paths ───────────────────────────────────────────────
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv

_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
for _candidate in [_nb_dir / '.env', _nb_dir.parent / '.env', Path('.env')]:
    if _candidate.exists():
        load_dotenv(_candidate)
        print(f'Loaded .env from {_candidate}')
        break
else:
    print('No .env found — using Colab defaults')

REPO_DIR   = os.getenv('REPO_DIR',   '/content/Yolo-ST-GCN')
BRANCH     = os.getenv('BRANCH',     'refactor-1')
GYM288_PKL = os.getenv('GYM288_PKL', '/content/Gym288-skeleton/gym_288_skeleton.pkl')
GYM99_PKL  = os.getenv('GYM99_PKL',  '/content/Gym99-from-Gym288/gym99_from_gym288.pkl')
OUT_BASE   = os.getenv('OUT_DIR',    'outputs/ensemble')

APPARATUS_LIST   = ['VT', 'FX', 'BB', 'UB']
APPARATUS_RANGES = {'VT': (0, 5), 'FX': (6, 40), 'BB': (41, 73), 'UB': (74, 98)}
EXPERT_DIRS      = {ap: f'{OUT_BASE}/expert_{ap}' for ap in APPARATUS_LIST}
FEATURES_DIR     = f'{OUT_BASE}/features'
META_DIR         = f'{OUT_BASE}/meta_learner'

print(f'REPO_DIR   = {REPO_DIR}')
print(f'GYM288_PKL = {GYM288_PKL}')
print(f'GYM99_PKL  = {GYM99_PKL}')
print(f'OUT_BASE   = {OUT_BASE}')

In [ ]:
# ── Cell 2: Repo setup ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Download Gym288 dataset ───────────────────────────────────────────
if Path(GYM288_PKL).exists():
    print(f'Gym288 pickle found: {GYM288_PKL}')
else:
    print('Downloading from HuggingFace...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'], check=True)
    from huggingface_hub import snapshot_download
    download_dir = Path(GYM288_PKL).parent
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
    pkl_candidates = sorted(download_dir.rglob('*.pkl'))
    if not pkl_candidates:
        raise FileNotFoundError('No .pkl found after Gym288 download.')
    GYM288_PKL = str(pkl_candidates[0])
    print(f'Downloaded: {GYM288_PKL}')

In [ ]:
# ── Cell 4: Train Expert VT — 6 classes (Clabel 0-5) ──
import sys, importlib

sys.argv = [
    'train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             EXPERT_DIRS['VT'],
    '--apparatus',           'VT',
    '--epochs',              '30',
    '--batch_size',          '64',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--bbox_norm',
    '--warmup_epochs',       '5',
    '--use_augment_feeder',
    '--use_two_stream',
    '--use_weighted_sampler',
    '--grad_clip_norm',      '1.0',
]

import scripts.train_gym99 as _train_script
importlib.reload(_train_script)
print(f'\n>>> Training Expert VT (6 classes) ...')
_train_script.main()
print(f'\n✅ Expert VT done.')


In [ ]:
# ── Evaluate Expert VT ───────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

from src.two_stream_stgcn import TwoStream_STGCN
from src.checkpointing import load_checkpoint
from src.gym99_dataset import build_gym99_data_tensors
from src.skeleton_utils import bbox_normalize, center_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ap = 'VT'
lo, hi = APPARATUS_RANGES[ap]
num_cls = hi - lo + 1

if 'GYM99_J_DATA' not in globals():
    print(f'Loading data into RAM for sequential evaluation...')
    j_data, b_data, labels, flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PKL, joint_spec_name='coco18',
        split='all', keep_unknown_split=False, return_bone_data=True
    )
    j_data = bbox_normalize(j_data)
    b_data = bbox_normalize(b_data)
    j_data = center_normalize(j_data, 17) # COCO18 center is 17
    globals()['GYM99_J_DATA'] = j_data
    globals()['GYM99_B_DATA'] = b_data
    globals()['GYM99_LABELS'] = labels.numpy().astype(int)
    globals()['GYM99_FLAGS']  = flags.numpy().astype(int)

j_data = GYM99_J_DATA
b_data = GYM99_B_DATA
labels_np = GYM99_LABELS
flags_np = GYM99_FLAGS

mask = (labels_np >= lo) & (labels_np <= hi)
val_mask = mask & (flags_np == 0)
val_j = torch.tensor(j_data[val_mask], dtype=torch.float32)
val_b = torch.tensor(b_data[val_mask], dtype=torch.float32)
val_y = torch.tensor(labels_np[val_mask] - lo, dtype=torch.long)

loader = DataLoader(TensorDataset(val_j, val_b, val_y), batch_size=64, shuffle=False)

weights_path = f"{EXPERT_DIRS[ap]}/stgcn_gym99_coco18_2s_expert_{ap}.pth"
model = TwoStream_STGCN(num_classes=num_cls, joint_spec='coco18').to(device)
load_checkpoint(weights_path, model)
model.eval()

preds = []
with torch.no_grad():
    for bj, bb, _ in loader:
        out = model(bj.to(device), bb.to(device))
        preds.extend(out.argmax(1).cpu().tolist())

gts = val_y.tolist()
acc = accuracy_score(gts, preds)
mf1 = f1_score(gts, preds, average='macro', zero_division=0)
print(f"\n{'='*40}\n[Val] {ap} - Acc: {acc:.4f} | Macro F1: {mf1:.4f}\n{'='*40}")

cm = confusion_matrix(gts, preds, labels=list(range(num_cls)))
plt.figure(figsize=(max(5, num_cls*0.35), max(4, num_cls*0.35)))
sns.heatmap(cm, annot=num_cls <= 20, fmt='d', cmap='Blues',
            xticklabels=list(range(num_cls)), yticklabels=list(range(num_cls)),
            linewidths=0.3, linecolor='#e0e0e0', cbar=False)
plt.title(f"Confusion Matrix: Expert {ap}", fontsize=13, fontweight='bold', pad=10)
plt.ylabel('True Class', fontsize=11)
plt.xlabel('Predicted Class', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 5: Train Expert FX — 35 classes (Clabel 6-40) ──
import sys, importlib

sys.argv = [
    'train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             EXPERT_DIRS['FX'],
    '--apparatus',           'FX',
    '--epochs',              '50',
    '--batch_size',          '128',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--bbox_norm',
    '--warmup_epochs',       '5',
    '--use_augment_feeder',
    '--use_two_stream',
    '--use_weighted_sampler',
    '--grad_clip_norm',      '1.0',
]

import scripts.train_gym99 as _train_script
importlib.reload(_train_script)
print(f'\n>>> Training Expert FX (35 classes) ...')
_train_script.main()
print(f'\n✅ Expert FX done.')


In [ ]:
# ── Evaluate Expert FX ───────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

from src.two_stream_stgcn import TwoStream_STGCN
from src.checkpointing import load_checkpoint
from src.gym99_dataset import build_gym99_data_tensors
from src.skeleton_utils import bbox_normalize, center_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ap = 'FX'
lo, hi = APPARATUS_RANGES[ap]
num_cls = hi - lo + 1

if 'GYM99_J_DATA' not in globals():
    print(f'Loading data into RAM for sequential evaluation...')
    j_data, b_data, labels, flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PKL, joint_spec_name='coco18',
        split='all', keep_unknown_split=False, return_bone_data=True
    )
    j_data = bbox_normalize(j_data)
    b_data = bbox_normalize(b_data)
    j_data = center_normalize(j_data, 17) # COCO18 center is 17
    globals()['GYM99_J_DATA'] = j_data
    globals()['GYM99_B_DATA'] = b_data
    globals()['GYM99_LABELS'] = labels.numpy().astype(int)
    globals()['GYM99_FLAGS']  = flags.numpy().astype(int)

j_data = GYM99_J_DATA
b_data = GYM99_B_DATA
labels_np = GYM99_LABELS
flags_np = GYM99_FLAGS

mask = (labels_np >= lo) & (labels_np <= hi)
val_mask = mask & (flags_np == 0)
val_j = torch.tensor(j_data[val_mask], dtype=torch.float32)
val_b = torch.tensor(b_data[val_mask], dtype=torch.float32)
val_y = torch.tensor(labels_np[val_mask] - lo, dtype=torch.long)

loader = DataLoader(TensorDataset(val_j, val_b, val_y), batch_size=64, shuffle=False)

weights_path = f"{EXPERT_DIRS[ap]}/stgcn_gym99_coco18_2s_expert_{ap}.pth"
model = TwoStream_STGCN(num_classes=num_cls, joint_spec='coco18').to(device)
load_checkpoint(weights_path, model)
model.eval()

preds = []
with torch.no_grad():
    for bj, bb, _ in loader:
        out = model(bj.to(device), bb.to(device))
        preds.extend(out.argmax(1).cpu().tolist())

gts = val_y.tolist()
acc = accuracy_score(gts, preds)
mf1 = f1_score(gts, preds, average='macro', zero_division=0)
print(f"\n{'='*40}\n[Val] {ap} - Acc: {acc:.4f} | Macro F1: {mf1:.4f}\n{'='*40}")

cm = confusion_matrix(gts, preds, labels=list(range(num_cls)))
plt.figure(figsize=(max(5, num_cls*0.35), max(4, num_cls*0.35)))
sns.heatmap(cm, annot=num_cls <= 20, fmt='d', cmap='Blues',
            xticklabels=list(range(num_cls)), yticklabels=list(range(num_cls)),
            linewidths=0.3, linecolor='#e0e0e0', cbar=False)
plt.title(f"Confusion Matrix: Expert {ap}", fontsize=13, fontweight='bold', pad=10)
plt.ylabel('True Class', fontsize=11)
plt.xlabel('Predicted Class', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 6: Train Expert BB — 33 classes (Clabel 41-73) ──
import sys, importlib

sys.argv = [
    'train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             EXPERT_DIRS['BB'],
    '--apparatus',           'BB',
    '--epochs',              '50',
    '--batch_size',          '128',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--bbox_norm',
    '--warmup_epochs',       '5',
    '--use_augment_feeder',
    '--use_two_stream',
    '--use_weighted_sampler',
    '--grad_clip_norm',      '1.0',
]

import scripts.train_gym99 as _train_script
importlib.reload(_train_script)
print(f'\n>>> Training Expert BB (33 classes) ...')
_train_script.main()
print(f'\n✅ Expert BB done.')


In [ ]:
# ── Evaluate Expert BB ───────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

from src.two_stream_stgcn import TwoStream_STGCN
from src.checkpointing import load_checkpoint
from src.gym99_dataset import build_gym99_data_tensors
from src.skeleton_utils import bbox_normalize, center_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ap = 'BB'
lo, hi = APPARATUS_RANGES[ap]
num_cls = hi - lo + 1

if 'GYM99_J_DATA' not in globals():
    print(f'Loading data into RAM for sequential evaluation...')
    j_data, b_data, labels, flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PKL, joint_spec_name='coco18',
        split='all', keep_unknown_split=False, return_bone_data=True
    )
    j_data = bbox_normalize(j_data)
    b_data = bbox_normalize(b_data)
    j_data = center_normalize(j_data, 17) # COCO18 center is 17
    globals()['GYM99_J_DATA'] = j_data
    globals()['GYM99_B_DATA'] = b_data
    globals()['GYM99_LABELS'] = labels.numpy().astype(int)
    globals()['GYM99_FLAGS']  = flags.numpy().astype(int)

j_data = GYM99_J_DATA
b_data = GYM99_B_DATA
labels_np = GYM99_LABELS
flags_np = GYM99_FLAGS

mask = (labels_np >= lo) & (labels_np <= hi)
val_mask = mask & (flags_np == 0)
val_j = torch.tensor(j_data[val_mask], dtype=torch.float32)
val_b = torch.tensor(b_data[val_mask], dtype=torch.float32)
val_y = torch.tensor(labels_np[val_mask] - lo, dtype=torch.long)

loader = DataLoader(TensorDataset(val_j, val_b, val_y), batch_size=64, shuffle=False)

weights_path = f"{EXPERT_DIRS[ap]}/stgcn_gym99_coco18_2s_expert_{ap}.pth"
model = TwoStream_STGCN(num_classes=num_cls, joint_spec='coco18').to(device)
load_checkpoint(weights_path, model)
model.eval()

preds = []
with torch.no_grad():
    for bj, bb, _ in loader:
        out = model(bj.to(device), bb.to(device))
        preds.extend(out.argmax(1).cpu().tolist())

gts = val_y.tolist()
acc = accuracy_score(gts, preds)
mf1 = f1_score(gts, preds, average='macro', zero_division=0)
print(f"\n{'='*40}\n[Val] {ap} - Acc: {acc:.4f} | Macro F1: {mf1:.4f}\n{'='*40}")

cm = confusion_matrix(gts, preds, labels=list(range(num_cls)))
plt.figure(figsize=(max(5, num_cls*0.35), max(4, num_cls*0.35)))
sns.heatmap(cm, annot=num_cls <= 20, fmt='d', cmap='Blues',
            xticklabels=list(range(num_cls)), yticklabels=list(range(num_cls)),
            linewidths=0.3, linecolor='#e0e0e0', cbar=False)
plt.title(f"Confusion Matrix: Expert {ap}", fontsize=13, fontweight='bold', pad=10)
plt.ylabel('True Class', fontsize=11)
plt.xlabel('Predicted Class', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 7: Train Expert UB — 25 classes (Clabel 74-98) ──
import sys, importlib

sys.argv = [
    'train_gym99.py',
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--out_dir',             EXPERT_DIRS['UB'],
    '--apparatus',           'UB',
    '--epochs',              '40',
    '--batch_size',          '128',
    '--lr',                  '0.001',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--bbox_norm',
    '--warmup_epochs',       '5',
    '--use_augment_feeder',
    '--use_two_stream',
    '--use_weighted_sampler',
    '--grad_clip_norm',      '1.0',
]

import scripts.train_gym99 as _train_script
importlib.reload(_train_script)
print(f'\n>>> Training Expert UB (25 classes) ...')
_train_script.main()
print(f'\n✅ Expert UB done.')


In [ ]:
# ── Evaluate Expert UB ───────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

from src.two_stream_stgcn import TwoStream_STGCN
from src.checkpointing import load_checkpoint
from src.gym99_dataset import build_gym99_data_tensors
from src.skeleton_utils import bbox_normalize, center_normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ap = 'UB'
lo, hi = APPARATUS_RANGES[ap]
num_cls = hi - lo + 1

if 'GYM99_J_DATA' not in globals():
    print(f'Loading data into RAM for sequential evaluation...')
    j_data, b_data, labels, flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PKL, joint_spec_name='coco18',
        split='all', keep_unknown_split=False, return_bone_data=True
    )
    j_data = bbox_normalize(j_data)
    b_data = bbox_normalize(b_data)
    j_data = center_normalize(j_data, 17) # COCO18 center is 17
    globals()['GYM99_J_DATA'] = j_data
    globals()['GYM99_B_DATA'] = b_data
    globals()['GYM99_LABELS'] = labels.numpy().astype(int)
    globals()['GYM99_FLAGS']  = flags.numpy().astype(int)

j_data = GYM99_J_DATA
b_data = GYM99_B_DATA
labels_np = GYM99_LABELS
flags_np = GYM99_FLAGS

mask = (labels_np >= lo) & (labels_np <= hi)
val_mask = mask & (flags_np == 0)
val_j = torch.tensor(j_data[val_mask], dtype=torch.float32)
val_b = torch.tensor(b_data[val_mask], dtype=torch.float32)
val_y = torch.tensor(labels_np[val_mask] - lo, dtype=torch.long)

loader = DataLoader(TensorDataset(val_j, val_b, val_y), batch_size=64, shuffle=False)

weights_path = f"{EXPERT_DIRS[ap]}/stgcn_gym99_coco18_2s_expert_{ap}.pth"
model = TwoStream_STGCN(num_classes=num_cls, joint_spec='coco18').to(device)
load_checkpoint(weights_path, model)
model.eval()

preds = []
with torch.no_grad():
    for bj, bb, _ in loader:
        out = model(bj.to(device), bb.to(device))
        preds.extend(out.argmax(1).cpu().tolist())

gts = val_y.tolist()
acc = accuracy_score(gts, preds)
mf1 = f1_score(gts, preds, average='macro', zero_division=0)
print(f"\n{'='*40}\n[Val] {ap} - Acc: {acc:.4f} | Macro F1: {mf1:.4f}\n{'='*40}")

cm = confusion_matrix(gts, preds, labels=list(range(num_cls)))
plt.figure(figsize=(max(5, num_cls*0.35), max(4, num_cls*0.35)))
sns.heatmap(cm, annot=num_cls <= 20, fmt='d', cmap='Blues',
            xticklabels=list(range(num_cls)), yticklabels=list(range(num_cls)),
            linewidths=0.3, linecolor='#e0e0e0', cbar=False)
plt.title(f"Confusion Matrix: Expert {ap}", fontsize=13, fontweight='bold', pad=10)
plt.ylabel('True Class', fontsize=11)
plt.xlabel('Predicted Class', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: Stage 2 — Feature Extraction ─────────────────────────────────────
# Each frozen two-stream expert backbone → 256-dim fused joint/bone vector.
# Vectors are L2-normalised before concatenation to prevent any expert dominating.
# Output: super_vectors (N, 1024), labels (N,), flags (N,)

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from src.gym99_dataset import build_gym99_data_tensors
from src.skeleton_utils import bbox_normalize
from src.two_stream_stgcn import TwoStream_STGCN
from src.checkpointing import load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('\nLoading full Gym99 tensors (train + val)...')
joint_data, bone_data, labels, flags, _, _ = build_gym99_data_tensors(
    dataset_path=GYM99_PKL,
    joint_spec_name='coco18',
    split='all',
    keep_unknown_split=False,
    return_bone_data=True,
)
joint_data = bbox_normalize(joint_data)
bone_data  = bbox_normalize(bone_data)
labels_np  = labels.numpy().astype('int64')
flags_np   = flags.numpy().astype('int64')
print(f'  Total={len(joint_data)}  train={int((flags_np==1).sum())}  val={int((flags_np==0).sum())}')

def extract_features(b_joint: torch.Tensor, b_bone: torch.Tensor, apparatus: str, batch_size: int = 64) -> np.ndarray:
    lo, hi = APPARATUS_RANGES[apparatus]
    # Note the _2s in the weights filename
    weights_path = Path(EXPERT_DIRS[apparatus]) / f'stgcn_gym99_coco18_2s_expert_{apparatus}.pth'
    model = TwoStream_STGCN(num_classes=hi - lo + 1, joint_spec='coco18').to(device)
    load_checkpoint(str(weights_path), model)
    model.eval()
    
    def get_stream_feat(batch, stream_model):
        n, c, t, v, m = batch.size()
        x = batch.permute(0, 4, 3, 1, 2).contiguous().view(n, m * v * c, t)
        x = stream_model.data_bn(x)
        x = x.view(n, m, v, c, t).permute(0, 1, 3, 4, 2).contiguous().view(n * m, c, t, v)
        for gcn in stream_model.st_gcn_networks:
            x = gcn(x)
        x = F.avg_pool2d(x, x.size()[2:])           # (N*M, 256, 1, 1)
        x = x.view(n, m, -1).mean(dim=1)            # (N, 256)
        return x

    parts = []
    loader = DataLoader(TensorDataset(b_joint, b_bone), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for (bj, bb) in loader:
            fj = get_stream_feat(bj.to(device), model.joint_stream)
            fb = get_stream_feat(bb.to(device), model.bone_stream)
            alpha = torch.sigmoid(model.alpha_logit)
            fused = alpha * fj + (1.0 - alpha) * fb
            parts.append(fused.cpu().numpy())
    return np.concatenate(parts, axis=0)

Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)
all_parts = []

for ap in APPARATUS_LIST:
    print(f'\nExtracting — Expert {ap}...')
    feats = extract_features(joint_data, bone_data, ap)      # (N, 256)
    norms = np.linalg.norm(feats, axis=1, keepdims=True).clip(min=1e-8)
    feats = feats / norms                                # L2-normalize
    all_parts.append(feats)
    print(f'  shape={feats.shape}  |  avg L2-norm={np.linalg.norm(feats, axis=1).mean():.4f}')

super_vectors = np.concatenate(all_parts, axis=1)        # (N, 1024)
np.save(f'{FEATURES_DIR}/super_vectors.npy', super_vectors)
np.save(f'{FEATURES_DIR}/labels.npy',        labels_np)
np.save(f'{FEATURES_DIR}/flags.npy',         flags_np)
print(f'\n✅ Saved super_vectors {super_vectors.shape} → {FEATURES_DIR}')


In [ ]:
# ── Cell 6: Stage 3 — Train Meta-Learner MLP ─────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Config
META_EPOCHS  = 100
META_LR      = 3e-4
META_BATCH   = 256
NUM_CLASSES  = 99
FEAT_DIM     = 1024

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sv     = np.load(f'{FEATURES_DIR}/super_vectors.npy')
labels = np.load(f'{FEATURES_DIR}/labels.npy')
flags  = np.load(f'{FEATURES_DIR}/flags.npy')

X_tr = torch.tensor(sv[flags==1],     dtype=torch.float32)
y_tr = torch.tensor(labels[flags==1], dtype=torch.long)
X_va = torch.tensor(sv[flags==0],     dtype=torch.float32)
y_va = torch.tensor(labels[flags==0], dtype=torch.long)
print(f'Train: {len(X_tr)}  Val: {len(X_va)}  Feature dim: {FEAT_DIM}')

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=META_BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_va, y_va), batch_size=META_BATCH, shuffle=False)

class MetaMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEAT_DIM, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)

meta_model = MetaMLP().to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(meta_model.parameters(), lr=META_LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=META_EPOCHS, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
Path(META_DIR).mkdir(parents=True, exist_ok=True)

for epoch in range(1, META_EPOCHS + 1):
    meta_model.train()
    t_loss = t_correct = t_total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = meta_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(meta_model.parameters(), 1.0)
        optimizer.step()
        t_loss    += loss.item()
        t_correct += (out.argmax(1) == yb).sum().item()
        t_total   += len(yb)
    scheduler.step()

    meta_model.eval()
    v_loss = v_correct = v_total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            out = meta_model(xb)
            v_loss    += criterion(out, yb).item()
            v_correct += (out.argmax(1) == yb).sum().item()
            v_total   += len(yb)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    history['train_loss'].append(t_loss / len(train_dl))
    history['val_loss'].append(v_loss / len(val_dl))
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(meta_model.state_dict(), f'{META_DIR}/meta_mlp_best.pth')

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{META_EPOCHS}  '
              f'train_loss={history["train_loss"][-1]:.4f}  train_acc={t_acc:.4f}  '
              f'val_loss={history["val_loss"][-1]:.4f}  val_acc={v_acc:.4f}')

torch.save(meta_model.state_dict(), f'{META_DIR}/meta_mlp_last.pth')
print(f'\n✅ Training done.  Best val_acc = {best_val_acc:.4f}')

In [ ]:
# ── Cell 7: Training curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Meta-Learner MLP — Training History', fontsize=13, fontweight='bold')

axes[0].plot(ep, history['train_loss'], label='Train', color='#e74c3c', linewidth=2)
axes[0].plot(ep, history['val_loss'],   label='Val',   color='#3498db', linewidth=2)
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_acc'], label='Train', color='#e74c3c', linewidth=2)
axes[1].plot(ep, history['val_acc'],   label='Val',   color='#3498db', linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{META_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8: Evaluation & Confusion Matrices ───────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
meta_model.load_state_dict(torch.load(f'{META_DIR}/meta_mlp_best.pth', map_location=device))
meta_model.eval()

sv     = np.load(f'{FEATURES_DIR}/super_vectors.npy')
labels = np.load(f'{FEATURES_DIR}/labels.npy')
flags  = np.load(f'{FEATURES_DIR}/flags.npy')

def infer(X_np, y_np, split_name):
    loader = DataLoader(TensorDataset(torch.tensor(X_np, dtype=torch.float32),
                                      torch.tensor(y_np, dtype=torch.long)),
                        batch_size=256, shuffle=False)
    preds, gts = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.extend(meta_model(xb.to(device)).argmax(1).cpu().tolist())
            gts.extend(yb.tolist())
    acc = accuracy_score(gts, preds)
    mf1 = f1_score(gts, preds, average='macro', zero_division=0)
    print(f'[{split_name}] Accuracy={acc:.4f}  Macro-F1={mf1:.4f}')
    return gts, preds

train_gt, train_pred = infer(sv[flags==1], labels[flags==1], 'TRAIN')
val_gt,   val_pred   = infer(sv[flags==0], labels[flags==0], 'VAL')

def save_cm(gt, preds, title, path, n_cls=99):
    cm = confusion_matrix(gt, preds, labels=list(range(n_cls)))
    fig, ax = plt.subplots(figsize=(22, 18))
    sns.heatmap(cm, ax=ax, cmap='Blues',
                xticklabels=list(range(n_cls)), yticklabels=list(range(n_cls)),
                linewidths=0.3, linecolor='#e0e0e0',
                cbar_kws={'label': 'count'})
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.tick_params(axis='both', labelsize=6)
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)

save_cm(train_gt, train_pred, 'Confusion Matrix — Train (Meta-Learner)', f'{META_DIR}/cm_train.png')
save_cm(val_gt,   val_pred,   'Confusion Matrix — Val   (Meta-Learner)', f'{META_DIR}/cm_val.png')

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(24, 10))
axes[0].imshow(mpimg.imread(f'{META_DIR}/cm_train.png'))
axes[0].axis('off')
axes[0].set_title('Train Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].imshow(mpimg.imread(f'{META_DIR}/cm_val.png'))
axes[1].axis('off')
axes[1].set_title('Val Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-apparatus breakdown
print('\n── Per-apparatus Val breakdown ──')
for ap, (lo, hi) in APPARATUS_RANGES.items():
    mask = [(g >= lo and g <= hi) for g in val_gt]
    if not any(mask):
        continue
    g = [val_gt[i]   for i in range(len(val_gt))   if mask[i]]
    p = [val_pred[i] for i in range(len(val_pred)) if mask[i]]
    print(f'  {ap} ({lo:2d}-{hi:2d})  n={len(g):4d}  '
          f'acc={accuracy_score(g, p):.4f}  macro-f1={f1_score(g, p, average="macro", zero_division=0):.4f}')